# Step 1: Embedding-Based Anomaly Score (TimesFM 2.5)

**What this does:** for each window of the series, get the hidden state from a middle layer of TimesFM 2.5, then score how far it is from the cloud of normal hidden states using Mahalanobis distance. Big distance = anomaly.

**Why:** residual+IQR misses anomalies the forecaster predicts accurately and false-alarms on rare-but-normal patterns. Embeddings sidestep both.

**Requires:** GPU, internet (one-time model download ~800MB), ~6GB VRAM.

References: THEMIS (arXiv 2510.03911), TimeRep (arXiv 2509.12650).


## 1. Install and import

In [ ]:
# %pip install -q timesfm torch numpy pandas matplotlib scikit-learn


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import timesfm

assert torch.cuda.is_available(), "GPU required for TimesFM 2.5"
DEVICE = "cuda"
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print("torch:", torch.__version__, "| cuda:", torch.cuda.get_device_name(0))


## 2. Load your data

Replace `load_data()` with your real loader. Required columns:
- `value` — the numerical series
- `is_true_anomaly` — boolean labels (only used for evaluation; remove for unsupervised use)


In [ ]:
def load_data(path: str | None = None) -> pd.DataFrame:
    if path is not None:
        return pd.read_csv(path, parse_dates=["timestamp"])
    # Fallback synthetic data so the notebook is runnable as-is. Delete once you have real data.
    rng = np.random.default_rng(SEED)
    n, period = 4000, 144
    t = np.arange(n)
    s = 0.0005*t + np.sin(2*np.pi*t/period) + 0.4*np.sin(2*np.pi*t/(period*7)) + rng.normal(0, 0.3, n)
    is_anom = np.zeros(n, dtype=bool)
    idx = rng.choice(np.arange(period*2, n-period), 40, replace=False)
    for i in idx:
        kind = rng.choice(["spike","drop","drift"])
        if kind=="spike": s[i] += rng.uniform(3,5)
        elif kind=="drop": s[i] -= rng.uniform(3,5)
        else: s[i:i+10] += rng.uniform(1.5,3)
        is_anom[i] = True
    return pd.DataFrame({
        "timestamp": pd.date_range("2025-01-01", periods=n, freq="10min"),
        "value": s, "is_true_anomaly": is_anom,
    })

df = load_data()                # <- pass your CSV path here
print(df.shape, "| anomalies:", df.is_true_anomaly.sum())


## 3. Load TimesFM 2.5 and hook a middle layer

The forward hook captures the hidden state of the chosen transformer block whenever the model runs. We use **layer 10 of 20** by default — middle layers give better representations than the final layer (TimeRep finding). Sweep this later: try 8, 10, 12.


In [ ]:
CTX_LEN   = 512        # context window per embedding. TimesFM 2.5 supports up to 16384.
LAYER_IDX = 10         # which transformer block's hidden state to use

tfm = timesfm.TimesFm_2p5(
    hparams=timesfm.TimesFmHparams(
        backend=DEVICE,
        per_core_batch_size=32,
        horizon_len=64,
        context_len=CTX_LEN,
    ),
    checkpoint=timesfm.TimesFmCheckpoint(
        huggingface_repo_id="google/timesfm-2.5-200m-pytorch"
    ),
)
tfm.model.eval()
for p in tfm.model.parameters(): p.requires_grad_(False)

# Forward hook. Captures hidden state at LAYER_IDX every forward pass.
_cache = {}
def _hook(module, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    _cache["h"] = h.detach()        # shape [B, T_patches, D]

# Path is `stacked_transformer.layers[i]` on TimesFM 2.5. If a new version
# moves things, `print(tfm.model)` and update the path.
target_block = tfm.model.stacked_transformer.layers[LAYER_IDX]
handle = target_block.register_forward_hook(_hook)

D = _cache.get("h", None)
print("hook registered on layer", LAYER_IDX)


## 4. Embed all windows

Slide a window of length `CTX_LEN` with stride `STRIDE` across the series. For each window, push it through TimesFM, grab the cached hidden state, and mean-pool over the patch dimension to get a single vector per window.


In [ ]:
STRIDE = 32      # smaller = more windows, more compute. 32 is a reasonable default.
BATCH  = 32

def make_windows(values, ctx_len=CTX_LEN, stride=STRIDE):
    xs, ends = [], []
    for end in range(ctx_len, len(values), stride):
        xs.append(values[end - ctx_len:end])
        ends.append(end - 1)
    return np.stack(xs).astype(np.float32), np.array(ends)

@torch.no_grad()
def embed_batch(windows_np):
    '''windows_np: [B, CTX_LEN] float32. Returns [B, D] numpy.'''
    x = torch.from_numpy(windows_np).to(DEVICE)
    _ = tfm.model(x)
    h = _cache["h"]                       # [B, T_patches, D]
    return h.mean(dim=1).cpu().numpy()    # mean-pool patches -> [B, D]

def embed_all(values):
    w, ends = make_windows(values)
    out = []
    for i in range(0, len(w), BATCH):
        out.append(embed_batch(w[i:i+BATCH]))
    return np.vstack(out), ends

# Fit the "normal cloud" on the first 50% (assumed mostly normal).
split = len(df) // 2
train_emb, train_ends = embed_all(df.value.values[:split])
all_emb,   all_ends   = embed_all(df.value.values)
print("train_emb:", train_emb.shape, "| all_emb:", all_emb.shape, "| D:", train_emb.shape[1])


## 5. Mahalanobis score

Fit mean and covariance of the train cloud (with a small ridge for stability), then score every window by its Mahalanobis distance to that distribution.


In [ ]:
def fit_gaussian(X, ridge=1e-3):
    mu = X.mean(axis=0)
    Xc = X - mu
    cov = (Xc.T @ Xc) / max(len(X) - 1, 1)
    cov += ridge * np.eye(cov.shape[0])
    return mu, np.linalg.inv(cov)

def mahalanobis(X, mu, cov_inv):
    Xc = X - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", Xc, cov_inv, Xc))

mu, cov_inv = fit_gaussian(train_emb)
scores_w = mahalanobis(all_emb, mu, cov_inv)
print(f"score stats: min={scores_w.min():.2f}  med={np.median(scores_w):.2f}  max={scores_w.max():.2f}")


## 6. Align scores back to the time axis and visualize

In [ ]:
def align_to_series(ends, scores, n_total):
    aligned = np.full(n_total, np.nan)
    aligned[ends] = scores
    last_score = np.nan
    for i in range(n_total):
        if not np.isnan(aligned[i]): last_score = aligned[i]
        elif not np.isnan(last_score): aligned[i] = last_score
    return aligned

scores_series = align_to_series(all_ends, scores_w, len(df))

fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(df.value.values, lw=0.5)
ax[0].scatter(np.where(df.is_true_anomaly)[0], df.value[df.is_true_anomaly],
              c="red", s=10, label="true anomaly", zorder=3)
ax[0].set_ylabel("value"); ax[0].legend()
ax[1].plot(scores_series, lw=0.6, c="purple")
ax[1].scatter(np.where(df.is_true_anomaly)[0], scores_series[df.is_true_anomaly],
              c="red", s=10, zorder=3)
ax[1].set_ylabel("Mahalanobis score"); ax[1].set_xlabel("index")
plt.tight_layout(); plt.show()


## 7. Evaluate vs. residual+IQR baseline

Threshold the embedding score at its 95th percentile; compare against the residual+IQR baseline on precision / recall / F1.


In [ ]:
def evaluate(flags, truth):
    flags = flags.astype(bool); truth = truth.astype(bool)
    tp = int((flags & truth).sum()); fp = int((flags & ~truth).sum()); fn = int((~flags & truth).sum())
    prec = tp / max(tp+fp, 1); rec = tp / max(tp+fn, 1)
    f1 = 2*prec*rec / max(prec+rec, 1e-9)
    return {"precision": round(prec,3), "recall": round(rec,3), "f1": round(f1,3),
            "tp": tp, "fp": fp, "fn": fn}

# Baseline: residual + IQR (rolling-mean residual as a stand-in for any forecaster residual)
roll = pd.Series(df.value.values).rolling(window=144, center=True, min_periods=24).mean()
residual = df.value.values - roll.values
mask = np.isfinite(residual)
q1, q3 = np.percentile(residual[mask], 25), np.percentile(residual[mask], 75)
iqr = q3 - q1
flags_iqr = np.zeros(len(df), dtype=bool)
flags_iqr[mask] = (residual[mask] < q1 - 3*iqr) | (residual[mask] > q3 + 3*iqr)

# New: embedding Mahalanobis, top-5% as flags
finite_s = scores_series[np.isfinite(scores_series)]
flags_emb = np.isfinite(scores_series) & (scores_series >= np.percentile(finite_s, 95))

print("residual + IQR :", evaluate(flags_iqr,  df.is_true_anomaly.values))
print("embedding score:", evaluate(flags_emb,  df.is_true_anomaly.values))


## 8. Knobs to sweep

If embedding F1 is not clearly above residual+IQR:
1. **`LAYER_IDX`** — try 8, 12. Re-register the hook (remove the old one first: `handle.remove()`).
2. **`CTX_LEN`** — try 1024, 2048. Longer context = more seasonal info per embedding.
3. **`STRIDE`** — try 8 or 16 for denser coverage (slower).
4. **Threshold** — 95th percentile assumes ~5% anomaly rate. Adjust to your dataset's expected rate. (Step 2 replaces this with a calibrated p-value.)

Tell me your numbers when you've run it. Step 2 (conformal calibration) is next.
